# Introduzione

## Protocolli e formati

### Breve storia

La posta elettronica precede l'internet moderno. Il primo trasferimento di messaggi in rete avvenne nel **1971** su ARPANET (Ray Tomlinson, che scelse anche `@` come separatore tra utente e host).

I protocolli che ancora oggi governano la posta elettronica furono standardizzati attraverso una serie di RFC:

| Anno | RFC | Cosa definisce |
|------|-----|----------------|
| 1982 | [RFC 821](https://www.rfc-editor.org/rfc/rfc821) | **SMTP** – Simple Mail Transfer Protocol (invio) |
| 1982 | [RFC 822](https://www.rfc-editor.org/rfc/rfc822) | **Formato del messaggio** – intestazioni, corpo, `From:`, `To:`, `Subject:` |
| 1988 | [RFC 1064](https://www.rfc-editor.org/rfc/rfc1064) → [RFC 3501](https://www.rfc-editor.org/rfc/rfc3501) | **IMAP** – Internet Message Access Protocol (accesso alla casella di posta) |
| 1991 | [RFC 1225](https://www.rfc-editor.org/rfc/rfc1225) → [RFC 1939](https://www.rfc-editor.org/rfc/rfc1939) | **POP3** – Post Office Protocol v3 (recupero semplificato) |
| 1992–1996 | [RFC 2045–2049](https://www.rfc-editor.org/rfc/rfc2045) | **MIME** – Multipurpose Internet Mail Extensions (allegati, HTML, codifiche) |
| 2008 | [RFC 5321](https://www.rfc-editor.org/rfc/rfc5321) | SMTP rivisto (ancora in uso) |
| 2008 | [RFC 5322](https://www.rfc-editor.org/rfc/rfc5322) | Formato del messaggio rivisto (ancora in uso) |

#### Lo stack a tre protocolli

<table>
<thead><tr><th>Il tuo programma</th><th>Direzione</th><th>Server di posta</th><th>Direzione</th><th>Programma del destinatario</th></tr></thead>
<tbody>
<tr><td>compone il messaggio</td><td>→ SMTP (porta 587/465) →</td><td>memorizza il messaggio nella casella</td><td>← IMAP (porta 993) o POP3 (porta 995) ←</td><td>legge la casella</td></tr>
</tbody>
</table>

- **SMTP** è un protocollo *push*: ci si connette a un server e si consegna un messaggio.
- **IMAP** è un protocollo *pull*: i messaggi risiedono sul server; li si interroga e gestisce in remoto.
- **POP3** è un protocollo pull più semplice che di solito scarica ed elimina i messaggi; largamente superato da IMAP.

### Formato del messaggio (RFC 5322 + MIME)

#### La busta: RFC 5322

Un messaggio email è testo semplice diviso in due parti da una riga vuota:

- **Intestazioni** — una sequenza di righe `Nome: valore`. Alcune sono obbligatorie (`From`, `To`, `Date`, `Message-ID`); la maggior parte è facoltativa.
- **Corpo** — tutto ciò che segue la riga vuota; nel caso più semplice, testo semplice.

Un messaggio minimo valido è il seguente:

```
From: alice@example.com
To: bob@example.com
Date: Tue, 03 Jun 2025 10:00:00 +0200
Message-ID: <abc123@example.com>
Subject: Hello

Hi Bob, this is the body.
```

I valori delle intestazioni che contengono caratteri non-ASCII (lettere accentate, emoji) devono essere codificati con le **encoded-words** dell'RFC 2047, es. `=?utf-8?q?caf=C3=A9?=`. Python gestisce questo in modo trasparente.

#### MIME: aggiungere struttura al corpo

RFC 5322 da solo supporta solo testo semplice. **MIME** (RFC 2045–2049) estende i messaggi per trasportare più tipi di contenuto — HTML, immagini, allegati — suddividendo il corpo in **parti**, ciascuna con le proprie intestazioni.

L'intestazione chiave è `Content-Type`. Quando un messaggio (o una parte) contiene sotto-parti, il suo tipo è `multipart/subtype` e una stringa `boundary` delimita le parti:

```
Content-Type: multipart/mixed; boundary="BOUNDARY"

--BOUNDARY
Content-Type: text/plain

Il corpo di testo.
--BOUNDARY
Content-Type: application/pdf
Content-Disposition: attachment; filename="report.pdf"
Content-Transfer-Encoding: base64

JVBERi0xLj...
--BOUNDARY--
```

`Content-Transfer-Encoding` (`base64` o `quoted-printable`) gestisce i dati binari all'interno di un protocollo basato su testo.

#### I tre sottotipi multipart

Il sottotipo `multipart` controlla come un client di posta *interpreta* le parti, non come vengono serializzate (il formato via rete è lo stesso).

| Sottotipo | Semantica | Uso tipico |
|-----------|-----------|------------|
| `multipart/mixed` | Le parti sono elementi indipendenti | Testo del corpo + allegati |
| `multipart/alternative` | Le parti sono rappresentazioni alternative dello **stesso** contenuto; il client sceglie la migliore che riesce a visualizzare | Versione plain text + HTML della stessa email |
| `multipart/related` | Le parti formano un unico documento composito; la parte radice fa riferimento alle altre tramite `Content-ID` | Corpo HTML + immagini inline (`<img src="cid:logo@x">`) |

I messaggi reali spesso **annidano** questi tipi. Una struttura comune per un'email HTML con allegato è:

<table>
<tr><td><b>multipart/mixed</b></td><td>livello superiore</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;└ <b>multipart/alternative</b></td><td>il corpo in due forme</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├ text/plain</td><td>fallback</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└ multipart/related</td><td>HTML + risorse inline</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;├ text/html</td><td>il corpo HTML</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└ image/png (Content-ID: logo)</td><td>immagine inline</td></tr>
<tr><td>&nbsp;&nbsp;&nbsp;└ application/pdf</td><td>allegato</td></tr>
</table>

### A mani nude: dialogo TCP grezzo

SMTP e IMAP sono protocolli di testo orientati alla riga. Per dimostrarlo, le due celle seguenti aprono un socket TCP semplice e digitano ogni byte manualmente — nessun `smtplib`, nessun `imaplib`, nessuna libreria email di alcun tipo.

Gli helper sono volutamente minimali:
- `recv()` legge le righe dal server fino a quando non arriva una riga non di continuazione e stampa ciascuna preceduta da `S:`.
- `send()` aggiunge `\r\n`, invia i byte e stampa la riga preceduta da `C:`.

In [ ]:
import socket, base64

def send(sock, text):
    print('C:', text)
    sock.sendall((text + '\r\n').encode())

def recv(f, tag=None):
    while True: 
        line = f.readline().decode(errors='replace').rstrip('\r\n')
        print('S:', line)
        if tag is None:
            if len(line) < 4 or line[3] != '-':  # SMTP: stop at non-continuation line
                break
        else:
            if line.startswith(tag):              # IMAP: stop at tagged response
                break

#### SMTP

Il tipico scambio con un server SMTP è:

1. Connessione TCP
2. Il server saluta con `220`
3. Il client invia `EHLO` — il server risponde con le proprie capacità
4. (opzionale) Aggiornamento a TLS con `STARTTLS`
5. `AUTH` — autenticazione
6. `MAIL FROM` / `RCPT TO` / `DATA` — consegna del messaggio
7. `QUIT`

> **Nota:** il server locale GreenMail non richiede autenticazione, quindi il passo 5 viene omesso.

In [ ]:
with socket.create_connection(('localhost', 3025)) as sock, sock.makefile('rb') as f:
    recv(f)
    send(sock, 'EHLO notebook');
    recv(f)
    send(sock, 'MAIL FROM:<alice@example.com>');    
    recv(f)
    send(sock, 'RCPT TO:<rec@goo.bar>');
    recv(f)
    send(sock, 'DATA');
    recv(f)
    for line in [
        'From: alice@example.com', 
        'To: rec@goo.bar',
        'Subject: Bare hands', 
        '',
        'No libraries. No frameworks. Just TCP and text.', 
        '.']:
        send(sock, line)
    recv(f)
    send(sock, 'QUIT');

#### IMAP

IMAP è un protocollo *stateful*: dopo la connessione ci si trova sempre in uno di vari stati (non autenticato → autenticato → selezionato). La maggior parte delle operazioni ha senso solo nello stato *selected* (dopo aver eseguito `SELECT` su una casella).

Concetti chiave:

| Concetto | Significato |
|----------|-------------|
| **Mailbox** | Una cartella sul server (`INBOX`, `Sent`, `Drafts`, …) |
| **Sequence number** | Posizione di un messaggio nella casella attualmente selezionata — cambia man mano che i messaggi vengono aggiunti/eliminati |
| **UID** | Identificatore numerico stabile per un messaggio — sopravvive a expunge e riconnessioni |
| **Flag** | Stato per messaggio: `\Seen`, `\Answered`, `\Flagged`, `\Deleted`, `\Draft` |
| **SEARCH** | Filtraggio lato server per data, flag, intestazione, dimensione, … |
| **FETCH** | Recupera parte o tutto di un messaggio (`RFC822` = formato wire completo) |

Recuperare un messaggio **non** lo contrassegna come `\Seen` a meno che non si recuperi esplicitamente `BODY[]` (senza `.PEEK`). Usare `BODY.PEEK[]` è sicuro per l'ispezione in sola lettura.

In [ ]:

with socket.create_connection(('localhost', 3143)) as sock, sock.makefile('rb') as f:
    n = iter(range(1, 100))
    def cmd(text):
        tag = f'A{next(n):03d}'
        send(sock, f'{tag} {text}')
        recv(f, tag)

    recv(f)
    cmd('LOGIN rec@goo.bar recp')
    cmd('SELECT INBOX')
    cmd('SEARCH ALL')
    cmd('FETCH 1 (BODY.PEEK[HEADER.FIELDS (FROM SUBJECT)])')
    cmd('LOGOUT')

S: * OK IMAP4rev1 Server GreenMail v2.1.8 ready
C: A001 LOGIN rec@goo.bar recp
S: A001 OK LOGIN completed.
C: A002 SELECT INBOX
S: * FLAGS (\Answered \Deleted \Draft \Flagged \Seen)
S: * 4 EXISTS
S: * 0 RECENT
S: * OK [UIDVALIDITY 1780468883]
S: * OK [UIDNEXT 5]
S: * OK No messages unseen
S: * OK [PERMANENTFLAGS (\Answered \Deleted \Draft \Flagged \Seen \*)]
S: A002 OK [READ-WRITE] SELECT completed.
C: A003 SEARCH ALL
S: * SEARCH 1 2 3 4
S: A003 OK SEARCH completed.
C: A004 FETCH 1 (BODY.PEEK[HEADER.FIELDS (FROM SUBJECT)])
S: * 1 FETCH (BODY[HEADER.FIELDS (FROM SUBJECT)] {44}
S: Subject: A simple email
S: From: snd@foo.bar
S: )
S: A004 OK FETCH completed.
C: A005 LOGOUT
S: * BYE IMAP4rev1 Server logging out
S: A005 OK LOGOUT completed.
